
# DDPM vs. VAE: CIFAR-10 Model Comparison

Loads a trained conditional **DDPM** checkpoint and a trained conditional **VAE**
checkpoint, and compares them on:

1. Overall FID / Inception Score (mixed classes, N generated samples per model)
2. Per-class FID / Inception Score, plotted side by side
3. A qualitative check: 10 random DDPM-generated images vs. their 5 nearest
   real CIFAR-10 images (pixel-space L2 distance), to eyeball whether the
   model is generating something new or effectively memorizing training images

**Note on model loading:** both models are rebuilt using the architecture config
*saved inside their own checkpoint* (`checkpoint["config"]`), not whatever
happens to be in `configs.yml` on disk right now. That way this notebook works
regardless of what configs.yml has been edited to since training, and can't hit
the classic "size mismatch" error from an architecture/checkpoint mismatch.


In [ ]:
import os
import random

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

from cifar10_dataset import CIFAR10Dataset, denormalize
from ConditionalDDPM import ConditionalDDPM
from diffusion_utils import GaussianDiffusion
from ConditionalVAE import ConditionalVAE
from metrics import compute_fid_and_is

CIFAR10_CLASS_NAMES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]


## Configuration

Edit the paths and sample counts below to match your setup.

**A note on sample counts:** FID is a biased estimator whose bias shrinks as
sample count grows -- estimates from only a couple hundred samples are noisy
and not directly comparable to numbers reported in papers (which typically use
10k-50k samples). `N_SAMPLES_OVERALL` and `N_SAMPLES_PER_CLASS` below default
to modest values so this notebook runs in a reasonable time; raise them for
more trustworthy numbers if you can afford the extra generation time (DDPM
sampling in particular is slow -- it runs the full reverse diffusion loop for
every batch).


In [ ]:
# ---- Paths: edit these to point at your checkpoints and data ----
DDPM_CHECKPOINT_PATH = "./checkpoints/ddpm_epoch_070.pt"
VAE_CHECKPOINT_PATH = "./checkpoints/vae_epoch_070.pt"
DATA_DIR = "/kaggle/input/datasets/pankrzysiu/cifar10-python"
DOWNLOAD_DATA = False  # True if DATA_DIR doesn't already contain cifar-10-batches-py

# ---- Evaluation settings ----
N_SAMPLES_OVERALL = 2000      # total generated samples per model for the overall FID/IS estimate
N_SAMPLES_PER_CLASS = 200     # generated samples per class for the per-class FID/IS breakdown
FID_IS_BATCH_SIZE = 48       # generation batch size used internally by compute_fid_and_is

# ---- Nearest-neighbor settings ----
NUM_GENERATED_FOR_NN = 10     # how many DDPM images to generate for the similarity search
NUM_NEIGHBORS = 5             # how many nearest real images to retrieve per generated image

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)


## Load CIFAR-10 (real images, used as the FID/IS reference and the nearest-neighbor search bank)


In [ ]:
# CIFAR10Dataset auto-detects the exact dataset-slug subfolder under DATA_DIR
# (see cifar10_dataset.py's _resolve_cifar_root) -- handles Kaggle's
# /kaggle/input/<slug>/cifar-10-batches-py layout automatically.
train_dataset = CIFAR10Dataset(root=DATA_DIR, train=True, augment=False, download=DOWNLOAD_DATA)
test_dataset = CIFAR10Dataset(root=DATA_DIR, train=False, augment=False, download=DOWNLOAD_DATA)

NUM_CLASSES = 10


def indices_by_class(dataset, num_classes):
    '''torchvision CIFAR10 stores plain python-int labels in .targets.'''
    targets = dataset.base_dataset.targets
    buckets = {c: [] for c in range(num_classes)}
    for idx, label in enumerate(targets):
        buckets[label].append(idx)
    return buckets


train_indices_by_class = indices_by_class(train_dataset, NUM_CLASSES)
test_indices_by_class = indices_by_class(test_dataset, NUM_CLASSES)

print(f"Train set: {len(train_dataset)} images | Test set: {len(test_dataset)} images")
for c in range(NUM_CLASSES):
    print(
        f"  class {c} ({CIFAR10_CLASS_NAMES[c]}): "
        f"{len(train_indices_by_class[c])} train / {len(test_indices_by_class[c])} test"
    )


## Load both models


In [ ]:
ddpm_checkpoint = torch.load(DDPM_CHECKPOINT_PATH, map_location=DEVICE)
ddpm_cfg = ddpm_checkpoint["config"]
ddpm_model_cfg = ddpm_cfg["MODEL"]

ddpm_model = ConditionalDDPM(
    num_classes=ddpm_model_cfg["NUM_CLASSES"],
    embedding_dim=ddpm_model_cfg["CLASS_EMBEDDING_DIM"],
    num_groups=ddpm_model_cfg["NUM_GROUPS"],
    channels_per_level=ddpm_model_cfg["CHANNELS_PER_LEVEL"],
    theta=ddpm_model_cfg["THETA"],
).to(DEVICE)
ddpm_model.load_state_dict(ddpm_checkpoint["model_state_dict"])
ddpm_model.eval()

diffusion = GaussianDiffusion(
    timesteps=ddpm_cfg["DIFFUSION"]["TIMESTEPS"],
    beta_start=ddpm_cfg["DIFFUSION"]["BETA_START"],
    beta_end=ddpm_cfg["DIFFUSION"]["BETA_END"],
    schedule=ddpm_cfg["DIFFUSION"]["BETA_SCHEDULE"],
    device=DEVICE,
)

IMAGE_SIZE = ddpm_model_cfg["IMAGE_SIDE_LENGTH"]
print(f"Loaded DDPM checkpoint from epoch {ddpm_checkpoint['epoch']} | image size {IMAGE_SIZE}x{IMAGE_SIZE}")

In [ ]:
vae_checkpoint = torch.load(VAE_CHECKPOINT_PATH, map_location=DEVICE)
vae_cfg = vae_checkpoint["config"]
vae_model_cfg = vae_cfg["MODEL"]

vae_model = ConditionalVAE(
    num_classes=vae_model_cfg["NUM_CLASSES"],
    embedding_dim=vae_model_cfg["CLASS_EMBEDDING_DIM"],
    num_groups=vae_model_cfg["NUM_GROUPS"],
    channels_per_level=vae_model_cfg["CHANNELS_PER_LEVEL"],
    latent_channels=vae_model_cfg["LATENT_CHANNELS"],
).to(DEVICE)
vae_model.load_state_dict(vae_checkpoint["model_state_dict"])
vae_model.eval()

VAE_LATENT_CHANNELS = vae_model_cfg["LATENT_CHANNELS"]
VAE_LATENT_SPATIAL = vae_model_cfg["IMAGE_SIDE_LENGTH"] // 8

print(
    f"Loaded VAE checkpoint from epoch {vae_checkpoint['epoch']} | "
    f"latent shape ({VAE_LATENT_CHANNELS}, {VAE_LATENT_SPATIAL}, {VAE_LATENT_SPATIAL})"
)

if vae_model_cfg["IMAGE_SIDE_LENGTH"] != IMAGE_SIZE:
    print(
        f"WARNING: DDPM image size ({IMAGE_SIZE}) and VAE image size "
        f"({vae_model_cfg['IMAGE_SIDE_LENGTH']}) differ -- comparisons below assume matching sizes."
    )


## Sampling functions

Both wrapped to the same interface -- `(batch_size, class_ids=None) -> (B, 3, H, W)
tensor in [-1, 1]` -- so the FID/IS code below doesn't need to know which model
it's talking to.


In [ ]:
@torch.no_grad()
def ddpm_sample(batch_size, class_ids=None):
    '''Generates `batch_size` DDPM samples via the full reverse diffusion loop.
    If class_ids is None, classes are drawn uniformly at random.'''
    if class_ids is None:
        class_ids = torch.randint(0, NUM_CLASSES, (batch_size,), device=DEVICE)
    elif not torch.is_tensor(class_ids):
        class_ids = torch.as_tensor(class_ids, device=DEVICE, dtype=torch.long)
    samples = diffusion.p_sample_loop(ddpm_model, (batch_size, 3, IMAGE_SIZE, IMAGE_SIZE), class_ids, DEVICE)
    return samples.clamp(-1.0, 1.0)


@torch.no_grad()
def vae_sample(batch_size, class_ids=None):
    '''Generates `batch_size` VAE samples by decoding random latents.
    If class_ids is None, classes are drawn uniformly at random.'''
    if class_ids is None:
        class_ids = torch.randint(0, NUM_CLASSES, (batch_size,), device=DEVICE)
    elif not torch.is_tensor(class_ids):
        class_ids = torch.as_tensor(class_ids, device=DEVICE, dtype=torch.long)
    z = torch.randn(batch_size, VAE_LATENT_CHANNELS, VAE_LATENT_SPATIAL, VAE_LATENT_SPATIAL, device=DEVICE)
    samples = vae_model.decode(z, class_ids)
    return samples.clamp(-1.0, 1.0)


def make_class_conditional_sample_fn(sample_fn, class_id):
    '''Wraps a (batch_size, class_ids=None)->images sampler into a plain
    (batch_size)->images closure fixed to one class, for compute_fid_and_is.'''
    def _fn(batch_size):
        class_ids = torch.full((batch_size,), class_id, device=DEVICE, dtype=torch.long)
        return sample_fn(batch_size, class_ids=class_ids)
    return _fn


## Overall FID / Inception Score (mixed classes, N samples)

Both models are compared against the same real reference set (the CIFAR-10 test split).


In [ ]:
real_eval_loader = DataLoader(test_dataset, batch_size=FID_IS_BATCH_SIZE, shuffle=True, num_workers=2)

print("Computing overall FID/IS for DDPM (this runs the full reverse diffusion loop -- can be slow)...")
ddpm_overall_fid, ddpm_overall_is_mean, ddpm_overall_is_std = compute_fid_and_is(
    ddpm_sample, real_eval_loader, num_samples=N_SAMPLES_OVERALL, device=DEVICE, batch_size=FID_IS_BATCH_SIZE,
)
print(f"DDPM overall: FID={ddpm_overall_fid:.3f} | IS={ddpm_overall_is_mean:.3f} +/- {ddpm_overall_is_std:.3f}")

print("Computing overall FID/IS for VAE...")
vae_overall_fid, vae_overall_is_mean, vae_overall_is_std = compute_fid_and_is(
    vae_sample, real_eval_loader, num_samples=N_SAMPLES_OVERALL, device=DEVICE, batch_size=FID_IS_BATCH_SIZE,
)
print(f"VAE overall:  FID={vae_overall_fid:.3f} | IS={vae_overall_is_mean:.3f} +/- {vae_overall_is_std:.3f}")

overall_results = {
    "DDPM": {"fid": ddpm_overall_fid, "is_mean": ddpm_overall_is_mean, "is_std": ddpm_overall_is_std},
    "VAE": {"fid": vae_overall_fid, "is_mean": vae_overall_is_mean, "is_std": vae_overall_is_std},
}


## Per-class FID / Inception Score

For each class, generates `N_SAMPLES_PER_CLASS` class-conditional samples and compares
them against that class's real test images only.


In [ ]:
if N_SAMPLES_PER_CLASS < 200:
    print(
        f"WARNING: N_SAMPLES_PER_CLASS={N_SAMPLES_PER_CLASS} is quite small -- FID estimates "
        f"get noisy/biased below a few hundred samples. Treat per-class numbers as directional, "
        f"not precise, unless you raise this."
    )

per_class_results = []  # list of dicts: model, class, class_name, fid, is_mean, is_std

for model_name, sample_fn in [("DDPM", ddpm_sample), ("VAE", vae_sample)]:
    for class_id in range(NUM_CLASSES):
        real_subset = Subset(test_dataset, test_indices_by_class[class_id])
        real_loader = DataLoader(real_subset, batch_size=min(FID_IS_BATCH_SIZE, len(real_subset)), shuffle=True)

        class_sample_fn = make_class_conditional_sample_fn(sample_fn, class_id)
        fid_value, is_mean, is_std = compute_fid_and_is(
            class_sample_fn, real_loader, num_samples=N_SAMPLES_PER_CLASS, device=DEVICE,
            batch_size=min(FID_IS_BATCH_SIZE, N_SAMPLES_PER_CLASS),
        )

        per_class_results.append({
            "model": model_name, "class": class_id, "class_name": CIFAR10_CLASS_NAMES[class_id],
            "fid": fid_value, "is_mean": is_mean, "is_std": is_std,
        })
        print(f"[{model_name}] {CIFAR10_CLASS_NAMES[class_id]:>10s}: FID={fid_value:.3f} | IS={is_mean:.3f} +/- {is_std:.3f}")

per_class_df = pd.DataFrame(per_class_results)
per_class_df


## Graphs


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

x = np.arange(NUM_CLASSES)
width = 0.35

ddpm_fid = per_class_df[per_class_df["model"] == "DDPM"].sort_values("class")["fid"].values
vae_fid = per_class_df[per_class_df["model"] == "VAE"].sort_values("class")["fid"].values

axes[0].bar(x - width / 2, ddpm_fid, width, label="DDPM")
axes[0].bar(x + width / 2, vae_fid, width, label="VAE")
axes[0].set_xticks(x)
axes[0].set_xticklabels(CIFAR10_CLASS_NAMES, rotation=45, ha="right")
axes[0].set_ylabel("FID (lower is better)")
axes[0].set_title("Per-class FID")
axes[0].legend()

ddpm_is_mean = per_class_df[per_class_df["model"] == "DDPM"].sort_values("class")["is_mean"].values
ddpm_is_std = per_class_df[per_class_df["model"] == "DDPM"].sort_values("class")["is_std"].values
vae_is_mean = per_class_df[per_class_df["model"] == "VAE"].sort_values("class")["is_mean"].values
vae_is_std = per_class_df[per_class_df["model"] == "VAE"].sort_values("class")["is_std"].values

axes[1].bar(x - width / 2, ddpm_is_mean, width, yerr=ddpm_is_std, capsize=3, label="DDPM")
axes[1].bar(x + width / 2, vae_is_mean, width, yerr=vae_is_std, capsize=3, label="VAE")
axes[1].set_xticks(x)
axes[1].set_xticklabels(CIFAR10_CLASS_NAMES, rotation=45, ha="right")
axes[1].set_ylabel("Inception Score (higher is better)")
axes[1].set_title("Per-class Inception Score")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

models = ["DDPM", "VAE"]
fid_values = [overall_results[m]["fid"] for m in models]
is_means = [overall_results[m]["is_mean"] for m in models]
is_stds = [overall_results[m]["is_std"] for m in models]

axes[0].bar(models, fid_values, color=["tab:blue", "tab:orange"])
axes[0].set_ylabel("FID (lower is better)")
axes[0].set_title(f"Overall FID (N={N_SAMPLES_OVERALL})")

axes[1].bar(models, is_means, yerr=is_stds, capsize=5, color=["tab:blue", "tab:orange"])
axes[1].set_ylabel("Inception Score (higher is better)")
axes[1].set_title(f"Overall Inception Score (N={N_SAMPLES_OVERALL})")

plt.tight_layout()
plt.show()

pd.DataFrame(overall_results).T


## Nearest-neighbor check: DDPM samples vs. real CIFAR-10

Generates 10 random DDPM images, then finds each one's 5 closest real training images
by pixel-space L2 distance. This is a coarse similarity metric -- it won't catch semantic
similarity the way a learned feature space would -- but it directly answers the practical
question: is the model reproducing something close to a specific training image, or
generating something genuinely new? Close matches with small distances are worth a closer
look; if every generated image's nearest neighbors are all roughly equally (and fairly
large) distances away, that's evidence against pixel-level memorization.


In [ ]:
gen_class_ids = torch.randint(0, NUM_CLASSES, (NUM_GENERATED_FOR_NN,), device=DEVICE)
generated_images = ddpm_sample(NUM_GENERATED_FOR_NN, class_ids=gen_class_ids)
generated_images_01 = denormalize(generated_images).cpu()

print(
    f"Generated {NUM_GENERATED_FOR_NN} DDPM images for classes: "
    f"{[CIFAR10_CLASS_NAMES[c] for c in gen_class_ids.cpu().tolist()]}"
)

In [ ]:
# Search bank = the full CIFAR-10 training set (what the model actually learned from).
real_bank_loader = DataLoader(train_dataset, batch_size=1000, shuffle=False, num_workers=2)

real_images_01_chunks = []
real_labels_chunks = []
for imgs, labels in real_bank_loader:
    real_images_01_chunks.append(denormalize(imgs))
    real_labels_chunks.append(labels)

real_images_01 = torch.cat(real_images_01_chunks, dim=0)
real_labels = torch.cat(real_labels_chunks, dim=0)
print(f"Real image search bank: {real_images_01.shape[0]} training images")

gen_flat = generated_images_01.view(NUM_GENERATED_FOR_NN, -1)
real_flat = real_images_01.view(real_images_01.shape[0], -1)

# Chunked distance computation so peak memory stays bounded regardless of bank size.
CHUNK = 5000
all_distances = torch.empty(NUM_GENERATED_FOR_NN, real_flat.shape[0])
for start in range(0, real_flat.shape[0], CHUNK):
    end = min(start + CHUNK, real_flat.shape[0])
    all_distances[:, start:end] = torch.cdist(gen_flat, real_flat[start:end])

neighbor_distances, neighbor_indices = torch.topk(all_distances, k=NUM_NEIGHBORS, largest=False, dim=1)

In [ ]:
fig, axes = plt.subplots(
    NUM_GENERATED_FOR_NN, NUM_NEIGHBORS + 1,
    figsize=(2.2 * (NUM_NEIGHBORS + 1), 2.2 * NUM_GENERATED_FOR_NN),
)

for row in range(NUM_GENERATED_FOR_NN):
    ax = axes[row, 0]
    ax.imshow(generated_images_01[row].permute(1, 2, 0).numpy())
    ax.set_title(f"Generated\n({CIFAR10_CLASS_NAMES[gen_class_ids[row].item()]})", fontsize=9)
    ax.axis("off")

    for col in range(NUM_NEIGHBORS):
        real_idx = neighbor_indices[row, col].item()
        dist = neighbor_distances[row, col].item()
        real_label = CIFAR10_CLASS_NAMES[real_labels[real_idx].item()]

        ax = axes[row, col + 1]
        ax.imshow(real_images_01[real_idx].permute(1, 2, 0).numpy())
        ax.set_title(f"#{col + 1}: {real_label}\nd={dist:.2f}", fontsize=8)
        ax.axis("off")

plt.suptitle("DDPM-generated (left) vs. 5 nearest real CIFAR-10 images (pixel-space L2)", y=1.001)
plt.tight_layout()
plt.show()


## Caveats

- **FID/IS at small N**: increase `N_SAMPLES_OVERALL` / `N_SAMPLES_PER_CLASS` for numbers
  you'd trust in a report; the defaults here favor a fast run over precision.
- **Per-class FID reference size**: the CIFAR-10 test split only has 1000 images per
  class, so per-class FID has a smaller, noisier real-image reference than the overall
  FID (which draws from the full 10k-image test set).
- **DDPM sampling is slow**: every generated image requires a full reverse diffusion
  loop (hundreds of steps), so the overall/per-class FID/IS cells above will take
  noticeably longer for DDPM than for VAE.
- **Nearest-neighbor distance is pixel-space L2**, not a learned similarity metric --
  it's a blunt but interpretable memorization check, not proof either way.
